# 00 — Data Ingestion

Collect the three raw evidence streams used by the project:

- SEC EDGAR → Form 4 insider transactions
- Alpha Vantage → daily OHLCV market data
- Marketaux → financial news

Each source is saved immediately to `data/raw/`. Existing snapshots are reused unless `FORCE_REFRESH=true`.


In [2]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env", override=True)

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from alpha_vantage_ingestion import fetch_daily_market_data  # noqa: E402
from marketaux_ingestion import fetch_financial_news  # noqa: E402
from sec_ingestion import fetch_form4_transactions  # noqa: E402
from shared import PATHS, ensure_directories, get_target_tickers  # noqa: E402

ensure_directories()

TICKERS = get_target_tickers()

ALPHA_VANTAGE_API_KEY = os.getenv("ALPHA_VANTAGE_API_KEY", "")
MARKETAUX_API_KEY = os.getenv("MARKETAUX_API_KEY", "")
SEC_USER_AGENT = os.getenv("SEC_USER_AGENT", "")

SEC_FORM4_LIMIT = int(os.getenv("SEC_FORM4_LIMIT", "20"))
NEWS_LIMIT_PER_TICKER = int(os.getenv("NEWS_LIMIT_PER_TICKER", "9"))
MARKETAUX_PAGE_SIZE = int(os.getenv("MARKETAUX_PAGE_SIZE", "3"))

FORCE_REFRESH = (
    os.getenv("FORCE_REFRESH", "false")
    .strip()
    .lower()
    in {"1", "true", "yes"}
)

RAW_OUTPUTS = {
    "sec": PATHS["raw"] / "sec_form4.csv",
    "market": PATHS["raw"] / "market_data.csv",
    "news": PATHS["raw"] / "news.csv",
}

print("Project root:", PROJECT_ROOT)
print("Target tickers:", TICKERS)
print("Force refresh:", FORCE_REFRESH)
print(".env exists:", (PROJECT_ROOT / ".env").exists())
print("SEC user agent loaded:", bool(SEC_USER_AGENT and "@" in SEC_USER_AGENT))
print("Alpha Vantage key loaded:", bool(ALPHA_VANTAGE_API_KEY))
print("Marketaux key loaded:", bool(MARKETAUX_API_KEY))

if not SEC_USER_AGENT or "@" not in SEC_USER_AGENT:
    raise ValueError("SEC_USER_AGENT is missing or invalid.")

if not ALPHA_VANTAGE_API_KEY:
    raise ValueError("ALPHA_VANTAGE_API_KEY is missing from .env.")

if not MARKETAUX_API_KEY:
    raise ValueError("MARKETAUX_API_KEY is missing from .env.")


Project root: C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System
Target tickers: ['AAPL']
Force refresh: True
.env exists: True
SEC user agent loaded: True
Alpha Vantage key loaded: True
Marketaux key loaded: True


## 1. SEC Form 4

In [3]:
sec_path = RAW_OUTPUTS["sec"]

if sec_path.exists() and not FORCE_REFRESH:
    print("Using existing SEC snapshot:", sec_path)
    sec_raw = pd.read_csv(sec_path)
else:
    print("Fetching SEC Form 4 data...")
    sec_raw = fetch_form4_transactions(
        tickers=TICKERS,
        user_agent=SEC_USER_AGENT,
        filings_per_ticker=SEC_FORM4_LIMIT,
    )
    sec_raw.to_csv(sec_path, index=False)
    print("Saved:", sec_path)

print("SEC rows:", len(sec_raw))
display(sec_raw.head())


Fetching SEC Form 4 data...
SEC: AAPL | CIK 0000320193 | 20 recent Form 4 filing(s)
SEC: parsed AAPL 0001140361-26-037020 (4 transaction row(s))
SEC: parsed AAPL 0001140361-26-036226 (1 transaction row(s))
SEC: parsed AAPL 0001140361-26-035636 (1 transaction row(s))
SEC: parsed AAPL 0001140361-26-035362 (1 transaction row(s))
SEC: parsed AAPL 0001140361-26-034741 (1 transaction row(s))
SEC: parsed AAPL 0001140361-26-033928 (1 transaction row(s))
SEC: parsed AAPL 0001140361-26-032884 (1 transaction row(s))
SEC: parsed AAPL 0001140361-26-025622 (3 transaction row(s))
SEC: parsed AAPL 0001140361-26-025620 (4 transaction row(s))
SEC: parsed AAPL 0001140361-26-023363 (2 transaction row(s))
SEC: parsed AAPL 0001140361-26-020871 (1 transaction row(s))
SEC: parsed AAPL 0001140361-26-020298 (3 transaction row(s))
SEC: parsed AAPL 0001140361-26-017175 (1 transaction row(s))
SEC: parsed AAPL 0001140361-26-015421 (6 transaction row(s))
SEC: parsed AAPL 0001140361-26-015420 (4 transaction row(s))
S

,ticker,company_name,owner_name,owner_title,transaction_date,filing_date,transaction_code,acquired_disposed,shares,price,accession_number,source_url
0,AAPL,Apple Inc.,Newstead Jennifer,"SVP, GC and Government Affairs",2026-09-15,2026-09-17,S,D,1438.0,330.19,0001140361-26-037020,https://www.sec.gov/Archives/edgar/data/320193...
1,AAPL,Apple Inc.,Newstead Jennifer,"SVP, GC and Government Affairs",2026-09-15,2026-09-17,M,A,30104.0,NaN,0001140361-26-037020,https://www.sec.gov/Archives/edgar/data/320193...
2,AAPL,Apple Inc.,Newstead Jennifer,"SVP, GC and Government Affairs",2026-09-15,2026-09-17,F,D,16228.0,331.34,0001140361-26-037020,https://www.sec.gov/Archives/edgar/data/320193...
3,AAPL,Apple Inc.,Newstead Jennifer,"SVP, GC and Government Affairs",2026-09-15,2026-09-17,M,D,30104.0,NaN,0001140361-26-037020,https://www.sec.gov/Archives/edgar/data/320193...
4,AAPL,Apple Inc.,Newstead Jennifer,"SVP, GC and Government Affairs",2026-09-08,2026-09-10,S,D,1438.0,317.23,0001140361-26-036226,https://www.sec.gov/Archives/edgar/data/320193...


## 2. Alpha Vantage Market Data

In [4]:
market_path = RAW_OUTPUTS["market"]

if market_path.exists() and not FORCE_REFRESH:
    print("Using existing market snapshot:", market_path)
    market_raw = pd.read_csv(market_path)
else:
    print("Fetching Alpha Vantage market data...")
    market_raw = fetch_daily_market_data(
        tickers=TICKERS,
        api_key=ALPHA_VANTAGE_API_KEY,
    )
    market_raw.to_csv(market_path, index=False)
    print("Saved:", market_path)

print("Market rows:", len(market_raw))
display(market_raw.head())


Fetching Alpha Vantage market data...
Saved: C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\raw\market_data.csv
Market rows: 100


,ticker,date,open,high,low,close,volume
0,AAPL,2026-09-18,337.9050,338.4900,332.5300,336.1300,86588203
1,AAPL,2026-09-17,334.7700,338.3400,330.1833,337.0000,36700225
2,AAPL,2026-09-16,332.5300,335.4800,330.7000,332.4100,35981000
3,AAPL,2026-09-15,330.1350,331.7800,328.3500,331.3400,31748183
4,AAPL,2026-09-14,334.7900,335.5000,331.3400,333.0800,39269147


## 3. Marketaux Financial News

In [5]:
news_path = RAW_OUTPUTS["news"]

if news_path.exists() and not FORCE_REFRESH:
    print("Using existing news snapshot:", news_path)
    news_raw = pd.read_csv(news_path)
else:
    print("Fetching Marketaux financial news...")
    news_raw = fetch_financial_news(
        tickers=TICKERS,
        api_token=MARKETAUX_API_KEY,
        limit_per_ticker=NEWS_LIMIT_PER_TICKER,
        page_size=MARKETAUX_PAGE_SIZE,
    )
    news_raw.to_csv(news_path, index=False)
    print("Saved:", news_path)

print("News rows:", len(news_raw))
display(news_raw.head())


Fetching Marketaux financial news...
Saved: C:\Users\admin\Desktop\AAI520\Investment-Research-Multi-Agent-System\data\raw\news.csv
News rows: 50


,ticker,published_at,title,description,content,source,url
0,AAPL,2026-09-20T18:56:59.000000Z,Astera Labs vs. Qualcomm: Which Semiconductor ...,One company is riding explosive AI connectivit...,The semiconductor sector is currently defined ...,finance.yahoo.com,https://finance.yahoo.com/markets/stocks/artic...
1,AAPL,2026-09-20T17:43:00.000000Z,Australian PM Albanese urges world to act for ...,Albanese urges US-China cooperation to address...,By Edward Johnson Australian Prime Minister An...,business-standard.com,https://www.business-standard.com/world-news/a...
2,AAPL,2026-09-20T17:05:00.000000Z,Wall Street Brunch: U.S.-China Summit In Spotl...,"Trump and Xi meet with trade, tariffs and AI o...",Getty Images\n\nDownload this episode on Apple...,seekingalpha.com,https://seekingalpha.com/article/4948147-wall-...
3,AAPL,2026-09-20T16:37:00.000000Z,"Prediction: Even With the $1,999 Price Tag, Ap...",Apple is on track to do something great that i...,"Apple's (NASDAQ: AAPL) new iPhones are here, a...",finance.yahoo.com,https://finance.yahoo.com/markets/stocks/artic...
4,AAPL,2026-09-20T16:25:40.000000Z,AAPL Looks 17.3% Overvalued on GF Value™ as In...,"On September 20, 2026, Australian Prime Minist...","On September 20, 2026, Australian Prime Minist...",gurufocus.com,https://www.gurufocus.com/news/9089254/aapl-lo...


## 4. Validate Raw Handoff

In [6]:
handoff_rows = []

for source, path in RAW_OUTPUTS.items():
    exists = path.exists()
    rows = len(pd.read_csv(path)) if exists else None

    handoff_rows.append(
        {
            "source": source,
            "path": str(path),
            "exists": exists,
            "rows": rows,
        }
    )

handoff = pd.DataFrame(handoff_rows)
display(handoff)

missing = handoff.loc[~handoff["exists"], "source"].tolist()

if missing:
    print("Missing raw source(s):", missing)
else:
    print("All raw sources are present.")
    print("Proceed to 01_data_preprocessing.ipynb.")


,source,path,exists,rows
0,sec,C:\Users\admin\Desktop\AAI520\Investment-Resea...,True,61
1,market,C:\Users\admin\Desktop\AAI520\Investment-Resea...,True,100
2,news,C:\Users\admin\Desktop\AAI520\Investment-Resea...,True,50


All raw sources are present.
Proceed to 01_data_preprocessing.ipynb.
